# Практика: Гибридный AI-оркестратор (Emulator)

**Цель:** Пройти путь запроса от пользователя до инференса, соблюдая правила безопасности, гибридного роутинга и учета затрат.

### Шаг 1: Установка и импорт компонентов
Мы используем асинхронные интерфейсы для работы с LLM.

In [6]:
import asyncio
import time
import random
import re
from dataclasses import dataclass
from typing import Dict, List, Optional

# Симуляция "библиотеки 2026" для защиты Supply Chain (Cosign/Kyverno)
def verify_model_signature(model_name: str):
    print(f"[Kyverno] Проверка подписи для {model_name}... ✅ OK (Sign: 0x88f2...)")
    return True

### Шаг 2: Слой безопасности (Presidio Masking)
Эмулируем шлюз деидентификации, который очищает данные перед отправкой в облако.

In [7]:
class SecurityGateway:
    def __init__(self):
        # Паттерны для маскирования (ФИО, Телефоны, Паспорта)
        self.patterns = {
            "PHONE": r"\b\d{10,11}\b",
            "PASSPORT": r"\b\d{4}\s\d{6}\b"
        }

    def anonymize(self, text: str) -> str:
        temp_text = text
        for label, pattern in self.patterns.items():
            temp_text = re.sub(pattern, f"<{label}_MASKED>", temp_text)
        return temp_text

# Тест
gateway = SecurityGateway()
print(gateway.anonymize("Мой паспорт 4510 123456, телефон 79001112233"))

Мой паспорт <PASSPORT_MASKED>, телефон <PHONE_MASKED>


### Шаг 3: Эмуляция инференс-движков (vLLM & YandexGPT)
Создаем классы, симулирующие поведение локального железа (RTX 4090) и облачного API.

In [8]:
@dataclass
class InferenceResponse:
    text: str
    tokens: int
    latency: float
    cost: float
    provider: str

class LocalInference:
    """Эмуляция On-Premise кластера (L40S/RTX 4090)"""
    def __init__(self):
        self.capacity = 0.8  # 80% вероятность успеха (симуляция нагрузки)
        self.fixed_cost_per_1k = 0.05 # Амортизация железа

    async def predict(self, prompt: str):
        if random.random() > self.capacity:
            raise Exception("503 Service Unavailable: GPU Cluster Overloaded")

        # Симуляция Speculative Decoding (быстрее обычного)
        start_time = time.time()
        await asyncio.sleep(0.5) # Имитация генерации
        return InferenceResponse(
            text=f"[Local Qwen-4] Ответ на: {prompt}",
            tokens=len(prompt.split()) * 2,
            latency=time.time() - start_time,
            cost=0.05,
            provider="On-Premise (RTX 4090)"
        )

class CloudInference:
    """Эмуляция Cloud API (YandexGPT5Pro)"""
    def __init__(self):
        self.cost_per_1k = 0.3 # Тариф 2026 года

    async def predict(self, prompt: str):
        start_time = time.time()
        await asyncio.sleep(0.8) # Облако чуть медленнее из-за сети
        return InferenceResponse(
            text=f"[YandexGPT] Ответ на: {prompt}",
            tokens=len(prompt.split()) * 2,
            latency=time.time() - start_time,
            cost=0.3,
            provider="Cloud (Yandex)"
        )

### Шаг 4: Слой маршрутизации LiteLLM (Hybrid Router)
Самый важный архитектурный узел: логика Fallback и роутинг.

In [9]:
class LiteLLM_Router:
    def __init__(self, local, cloud, security):
        self.local = local
        self.cloud = cloud
        self.security = security
        self.stats = {"on_prem": 0, "cloud": 0, "total_cost": 0.0}

    async def complete(self, prompt: str, sensitive_data: bool = False):
        # 1. Если данные чувствительны - маскируем
        clean_prompt = self.security.anonymize(prompt)

        # 2. Пытаемся идти в On-Prem (Priority 1)
        try:
            print(f"--- Запрос: {prompt[:30]}... ---")
            verify_model_signature("Qwen-3.5-35B")
            response = await self.local.predict(clean_prompt)
            self.stats["on_prem"] += 1

        # 3. Fallback паттерн (Priority 2)
        except Exception as e:
            print(f"⚠️ [Fallback] Локальный кластер упал: {e}. Переключаюсь на Облако...")
            response = await self.cloud.predict(clean_prompt)
            self.stats["cloud"] += 1

        self.stats["total_cost"] += response.cost
        return response

### Шаг 5: Финальный запуск и FinOps отчет
Запускаем пачку запросов и смотрим, как система выправляет экономику.

In [10]:
async def run_simulation():
    router = LiteLLM_Router(LocalInference(), CloudInference(), SecurityGateway())

    prompts = [
        "Как дела?",
        "Сделай отчет по паспорту 4400 111222", # ПДн
        "Рассчитай кредит для клиента 79991112233", # ПДн
        "Сколько будет 2+2?",
        "Напиши код на Python",
        "Проанализируй данные транзакций"
    ]

    for p in prompts:
        resp = await router.complete(p)
        print(f"Provider: {resp.provider} | Latency: {resp.latency:.2f}s | Cost: {resp.cost} руб.")
        print(f"Result: {resp.text}\n")

    print("="*30)
    print("AI FINOPS REPORT (Day 1)")
    print(f"Запросов On-Prem: {router.stats['on_prem']}")
    print(f"Запросов Cloud: {router.stats['cloud']}")
    print(f"Общий TCO: {router.stats['total_cost']:.2f} руб.")

    # Сравнение с чистым API
    api_only_cost = len(prompts) * 0.3
    saving = (1 - router.stats['total_cost']/api_only_cost) * 100
    print(f"Экономия vs Чистое API: {saving:.1f}%")

await run_simulation()

--- Запрос: Как дела?... ---
[Kyverno] Проверка подписи для Qwen-3.5-35B... ✅ OK (Sign: 0x88f2...)
Provider: On-Premise (RTX 4090) | Latency: 0.50s | Cost: 0.05 руб.
Result: [Local Qwen-4] Ответ на: Как дела?

--- Запрос: Сделай отчет по паспорту 4400 ... ---
[Kyverno] Проверка подписи для Qwen-3.5-35B... ✅ OK (Sign: 0x88f2...)
Provider: On-Premise (RTX 4090) | Latency: 0.50s | Cost: 0.05 руб.
Result: [Local Qwen-4] Ответ на: Сделай отчет по паспорту <PASSPORT_MASKED>

--- Запрос: Рассчитай кредит для клиента 7... ---
[Kyverno] Проверка подписи для Qwen-3.5-35B... ✅ OK (Sign: 0x88f2...)
Provider: On-Premise (RTX 4090) | Latency: 0.50s | Cost: 0.05 руб.
Result: [Local Qwen-4] Ответ на: Рассчитай кредит для клиента <PHONE_MASKED>

--- Запрос: Сколько будет 2+2?... ---
[Kyverno] Проверка подписи для Qwen-3.5-35B... ✅ OK (Sign: 0x88f2...)
Provider: On-Premise (RTX 4090) | Latency: 0.50s | Cost: 0.05 руб.
Result: [Local Qwen-4] Ответ на: Сколько будет 2+2?

--- Запрос: Напиши код на Python.

### Что мы продемонстрировали этим кодом:

1.  **Split MLOps / Hybrid Inference:** Модели работают в разных контурах. Мы приоритизируем локальное железо для снижения TCO.
2.  **Security (Presidio):** Код автоматически находит паспортные данные и маскирует их перед отправкой в "облачный" класс.
3.  **Supply Chain Protection:** Симуляция функции `verify_model_signature` показывает, как Kyverno проверяет веса перед инференсом.
4.  **Fallback Паттерн:** Если в `LocalInference` срабатывает ошибка (overload), `LiteLLM_Router` бесшовно перекидывает задачу в облако.
5.  **FinOps:** В конце мы видим реальную разницу в стоимости (On-prem обходится в 6 раз дешевле, чем API Яндекса).
6.  **Observability:** Мы логируем Latency и Provider для каждого запроса, как это делалось бы в Grafana.

**Задание для слушателей:** Измените `capacity` в `LocalInference` на `0.1` (имитация аварии в ЦОД) и посмотрите, как изменится стоимость TCO и поведение системы. Это и есть работа AI Архитектора — балансировать между надежностью и стоимостью.